# TP 2 : Agents avec Langchain

### Création d’un nouveau agent sans System Message

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama
model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
temperature=0
)
agent = create_agent(model=model)
question = HumanMessage(content="Quelle est la capitale de la lune ?")

response = agent.invoke({"messages": [question]}
)


### Agent avec System Message : un message de contrôle qui définit le  comportement global du modèle

In [ ]:
system_prompt = "Vous êtes un auteur de science-fiction ; créez une capitale à  la demande des utilisateurs." 

In [ ]:

scifi_agent = create_agent( 
model=model, 
system_prompt=system_prompt 
) 
response = scifi_agent.invoke( 
{"messages": [question]} 
) 

In [ ]:
print(response['messages'][-1].content)


### Agent avec Few-shot learning : une méthode où le modèle  apprend une nouvelle tâche ou classe à partir de quelques  exemples seulement 

In [ ]:
system_prompt = """ 
Vous êtes un auteur de science-fiction et vous devez créer une capitale spa tiale à la demande d'un utilisateur. 
Utilisateur : Quelle est la capitale de Mars ? 
Auteur : Marsialis 
Utilisateur : Quelle est la capitale de Vénus ? 
Auteur : Venusovia 
""" 

In [ ]:
scifi_agent = create_agent( 
model=model, 
system_prompt=system_prompt 
) 
response = scifi_agent.invoke( 
{"messages": [question]} 
) 

In [ ]:
print(response['messages'][-1].content)

### Agent avec réponse structurée: une sortie organisée selon un  format prédéfini, plutôt que du texte libre.

In [ ]:
system_prompt = """ 
Vous êtes un auteur de science-fiction et vous devez créer une capitale spa tiale à la demande d'un utilisateur. 
Veuillez respecter la structure ci-dessous. 
Nom : Nom de la capitale 
Localisation : Lieu où elle est située 
Ambiance : Description en 2 ou 3 mots 
Économie : Principaux secteurs d'activité 
"""


In [ ]:
scifi_agent = create_agent( 
model=model, 
system_prompt=system_prompt 
) 
response = scifi_agent.invoke( 
{"messages": [question]} 
) 


In [ ]:
print(response['messages'][-1].content)

### Agent avec réponse structurée en utilisant BaseModel : rendre la  réponse facile à exploiter automatiquement par un programme ou  un système.

In [ ]:
system_prompt = """
Vous êtes un auteur de science-fiction et vous devez créer une capitale spatiale à la demande d'un utilisateur.

Veuillez respecter la structure ci-dessous.

Nom : Nom de la capitale
Localisation : Lieu où elle est située
Ambiance : Description en 2 ou 3 mots
Economie : Principaux secteurs d'activité
"""

In [ ]:
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    nom: str
    Localisation: str
    Ambiance: str
    Economie: str


In [ ]:
agent = create_agent(
model=model,
system_prompt=system_prompt,
response_format=CapitalInfo 
) 
question = HumanMessage(content="Quelle est la capitale de la lune ?") 
response = agent.invoke( 
{"messages": [question]} 
) 
response["structured_response"] 


### Les tools (outils) : une fonction externe que le modèle peut  appeler pour effectuer une action précise qu’il ne peut pas faire  seul

#### Création du Tool 

In [ ]:
from langchain.tools import tool 


In [ ]:
@tool("meteo_capitale")
def meteo_capitale(ville: str) -> str:
    """
    Donne la météo d'une capitale (valeurs fixes pour test).
    Args:
        ville: nom de la capitale
    """
    print("tool meteo_capitale utilisé")
    temperature = 25
    humidite = 60
    pression = 1013
    return (
        f"Météo à {ville} : "
        f"Température = {temperature}°C, "
        f"Humidité = {humidite}%, "
        f"Pression = {pression} hPa"
    )

#### Ajout du Tool à l’agent 

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

model = ChatOllama(model="llama3.2:3b") 


In [ ]:
system_prompt = "Utilises les tools pour répondre aux questions" 
agent = create_agent( 
model=model, 
tools=[meteo_capitale], 
system_prompt=system_prompt, 
) 
question = HumanMessage(content="Quelle est la météo à Capitole lunaire ?") 
response = agent.invoke( 
{"messages": [question]} 
) 
print(response['messages'][-1].content) 


### Agent sans Web Search Tool 

In [ ]:
agent = create_agent( 
model=model, 
) 
question = HumanMessage(content="Vos connaissances en matière d'apprentissage  sont-elles à jour ?") 
response = agent.invoke( 
{"messages": [question]} 
) 
print(response['messages'][-1].content) 


### Agent avec Web Search Tool 


#### Sortie du Tool 


In [ ]:
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

tavily_client = TavilyClient()

@tool("web_search")
def web_search(query: str) -> Dict[str, Any]:
    """ Effectue une recherche sur le web en utilisant l'API de Tavily. Args: query: la requête de recherche """
    return tavily_client.search(query)

web_search.invoke("Qui est le Président de commune actuel de Marrakech ?")


#### Sortie de l’agent 

In [ ]:
agent = create_agent( 
model=model, 
tools=[web_search] 
) 
question = HumanMessage(content="Qui est le Président de commune actuel de  Marrakech ?") 
response = agent.invoke( 
{"messages": [question]}) 
print(response['messages'][-1].content) 


### Agent sans mémoire 


In [ ]:
agent = create_agent( 
model= model) 
question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un déve loppeur.") 
response = agent.invoke( 
{"messages": [question]}  
) 
print(response['messages'][-1].content) 
question = HumanMessage(content="Quel est mon métier ?") 
response = agent.invoke( 
{"messages": [question]}  
) 
print(response['messages'][-1].content)


### Agent avec mémoire

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  
agent = create_agent( 
model=model, 
checkpointer=InMemorySaver(),  
) 
question = HumanMessage(content="Bonjour, mon nom est Sami et je suis un déve loppeur.") 
config = {"configurable": {"thread_id": "1"}} 
response = agent.invoke( 
{"messages": [question]}, 
config,  
) 
question = HumanMessage(content="Quel est mon métier ?") 
response = agent.invoke( 
{"messages": [question]}, 
config,  
) 
print(response['messages'][-1].content) 


# TP : Chef personnel  

In [ ]:
import os
from typing import Any, Dict
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from tavily import TavilyClient

# Charger les variables d'environnement (.env)
load_dotenv()

# ==========================================
# 1. CONSTRUCTION DE L'OUTIL DE RECHERCHE
# ==========================================
def build_web_search_tool():
    """Construit un outil Tavily robuste avec gestion d'absence de clé API."""
    api_key = os.getenv("TAVILY_API_KEY")

    if not api_key:
        @tool("web_search")
        def web_search(query: str) -> Dict[str, Any]:
            """
            Recherche web indisponible tant que TAVILY_API_KEY n'est pas configurée.
            """
            return {
                "status": "error",
                "message": "La recherche web est indisponible. Ajoutez TAVILY_API_KEY dans le fichier .env."
            }
        return web_search

    tavily_client = TavilyClient(api_key=api_key)

    @tool("web_search")
    def web_search(query: str) -> Dict[str, Any]:
        """
        Recherche des informations culinaires sur le web (recettes, techniques, associations).
        Args:
            query: La question ou requête de recherche culinaire.
        """
        return tavily_client.search(query, max_results=3)

    return web_search

# ==========================================
# 2. CONFIGURATION ET CRÉATION DE L'AGENT
# ==========================================
def build_chef_agent():
    """Initialise l'agent avec son comportement global (System Message) et sa mémoire."""
    
    system_prompt = """
Tu es un chef cuisinier personnel intelligent.

Ton rôle:
- Aider l'utilisateur à cuisiner avec les ingrédients disponibles dans son réfrigérateur.
- Mémoriser ses préférences, allergies, régimes alimentaires et contraintes au fil de la discussion.
- Utiliser l'outil web_search uniquement si une information culinaire récente ou spécifique est nécessaire.
- Proposer des plats réalistes, simples et adaptés à la situation.

Règles de réponse:
- Réponds toujours en français.
- Tiens compte de l'historique complet de la conversation (mémoire).
- Privilégie les recettes faisables en moins de 20 minutes.
- N'invente pas d'ingrédients disponibles. Si tu proposes un ingrédient manquant, marque-le explicitement comme [Optionnel].
- Reste concis, précis et structure ta réponse ainsi:
  1. Plat proposé
  2. Pourquoi il convient (rappeler les préférences respectées)
  3. Ingrédients utilisés
  4. Étapes courtes de préparation
  5. Variante ou conseil du Chef
"""

    
    return create_agent(
        model=model,  # Utilise votre instance de modèle 'llama3.2:3b' déclarée plus haut
        tools=[build_web_search_tool()],
        system_prompt=system_prompt,
        checkpointer=InMemorySaver()  # Active la mémoire conversationnelle
    )

# ==========================================
# 3. FONCTION D'INTERACTION AVEC LA MÉMOIRE
# ==========================================
def ask_chef(agent, user_text: str, thread_id: str = "chef-session-tp") -> str:
    """Envoie un message à l'agent en préservant le contexte via l'identifiant de session."""
    response = agent.invoke(
        {"messages": [HumanMessage(content=user_text)]},
        {"configurable": {"thread_id": thread_id}}
    )
    return response["messages"][-1].content

# ==========================================
# 4. SCÉNARIO DE DÉMONSTRATION (AUTOMATIQUE)
# ==========================================
def run_chef_demo():
    """Exécute la séquence de test demandée par le TP pour valider le comportement."""
    chef_agent = build_chef_agent()
    
    
    scénario = [
        "Bonjour, je suis Najlaa. Je n'aime pas le piment et je suis végétarienne.",
        "Mémorise aussi que j'ai seulement 20 minutes pour cuisiner le soir.",
        "Dans mon réfrigérateur j'ai : des oeufs, des tomates, du fromage, des oignons et du pain. Propose-moi un plat adapté.",
        "Donne-moi une autre variante différente en utilisant le web si nécessaire pour vérifier une idée originale."
    ]

    print("=== DÉBUT DU TP AGENT : CHEF PERSONNEL ===\n")
    
    for i, prompt in enumerate(scénario, start=1):
        print(f" [Étape {i}] Utilisateur : {prompt}")
        réponse = ask_chef(chef_agent, prompt)
        print(f" Chef Personnel :\n{réponse}")
        print("-" * 60 + "\n")


run_chef_demo()